# AGRO AI — Plant Species Identification Training Pipeline
> **Smart Agriculture. Healthy Lives. Better Future.**

### Task: Botanical species classification (13 active species, scaling towards 10,000)

This notebook trains a deep learning transfer learning vision model using GPU acceleration on Google Colab.
Follow the numbered sections sequentially.

## 1. Environment Initialization & GPU Acceleration Verification

In [ ]:
import os
import sys
import json
import shutil
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import MobileNetV2
from sklearn.metrics import classification_report, confusion_matrix

print(f'TensorFlow Version: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'[OK] GPU Accelerator Detected: {gpus[0].name}')
else:
    print('[WARN] No GPU detected. In Colab, select: Runtime -> Change runtime type -> T4 GPU.')

## 2. Dataset Acquisition & Directory Setup

In [ ]:
# Define data directories
DATASET_DIR = Path('/content/datasets/plants')
EXPORT_DIR = Path('/content/exported_models')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_DIR.mkdir(parents=True, exist_ok=True)

print(f'Dataset path configured: {DATASET_DIR}')
print('If using Google Drive, you can mount via: from google.colab import drive; drive.mount("/content/drive")')

## 3. Dataset Inspection & Stratified Splitting

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20

# Data Augmentation & Normalization Generators
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.20,
    shear_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1.0 / 255.0)
print('[OK] Preprocessing and augmentation pipelines configured.')

## 4. Model Architecture & Transfer Learning Definition

In [ ]:
def build_transfer_learning_model(num_classes: int):
    base_model = MobileNetV2(
        input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
        include_top=False,
        weights='imagenet'
    )
    # Freeze early feature extractors
    for layer in base_model.layers[:-25]:
        layer.trainable = False

    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    predictions = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs=base_model.input, outputs=predictions)
    model.compile(
        optimizer=optimizers.Adam(learning_rate=3e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=min(3, num_classes), name='top_3_acc')]
    )
    return model

print('[OK] Architecture function defined with MobileNetV2 backbone.')

## 5. Model Training with Early Stopping & Checkpoints

In [ ]:
# Training execution placeholder
export_model_path = EXPORT_DIR / 'plant_species.keras'
training_callbacks = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    callbacks.ModelCheckpoint(str(export_model_path), monitor='val_accuracy', save_best_only=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6)
]
print(f'Model checkpoint target: {export_model_path}')

## 6. Evaluation, Confusion Matrix & Metrics

In [ ]:
def plot_training_curves(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(history.history['accuracy'], label='Train Accuracy')
    ax1.plot(history.history['val_accuracy'], label='Val Accuracy')
    ax1.set_title('Accuracy Trajectory')
    ax1.set_xlabel('Epoch')
    ax1.legend()
    ax1.grid(True)

    ax2.plot(history.history['loss'], label='Train Loss')
    ax2.plot(history.history['val_loss'], label='Val Loss')
    ax2.set_title('Loss Trajectory')
    ax2.set_xlabel('Epoch')
    ax2.legend()
    ax2.grid(True)
    plt.show()

## 7. Model & Metadata Export for AGRO AI Backend

In [ ]:
# Export class map metadata
class_metadata = {
    'task': 'plants',
    'model_filename': 'plant_species.keras',
    'classes': [
    "Apple (Malus domestica)",
    "Brinjal (Solanum melongena)",
    "Chilli / Pepper (Capsicum annuum)",
    "Corn / Maize (Zea mays)",
    "Cotton (Gossypium hirsutum)",
    "Cucumber (Cucumis sativus)",
    "Grape (Vitis vinifera)",
    "Mango (Mangifera indica)",
    "Potato (Solanum tuberosum)",
    "Rice / Paddy (Oryza sativa)",
    "Rose (Rosa rubiginosa)",
    "Tomato (Solanum lycopersicum)",
    "Wheat (Triticum aestivum)"
],
    'input_size': [224, 224, 3]
}
metadata_path = EXPORT_DIR / 'plant_species_metadata.json'
with open(metadata_path, 'w') as mf:
    json.dump(class_metadata, mf, indent=2)

print(f'[OK] Metadata written to: {metadata_path}')
print('\nTo deploy to AGRO AI:')
print('1. Download plant_species.keras and place into: backend/models/plant_species.keras')
print('2. Verify via: python backend/scripts/check_models.py')